In [ ]:
# ── Cell 1: Install ───────────────────────────────────────────────────────────
!pip install -q groq fastapi uvicorn nest_asyncio fitz tools

# ── Cell 2: Imports + Config ──────────────────────────────────────────────────
import re, json, time, math, statistics, asyncio, subprocess, threading
import uvicorn
import nest_asyncio
from groq import Groq
from fastapi import FastAPI, HTTPException
from fastapi.responses import JSONResponse
from pydantic import BaseModel
from typing import Optional

GROQ_API_KEY = ""
PORT = 8001  # different port from CV parser (8000)

try:
    from kaggle_secrets import UserSecretsClient
    GROQ_API_KEY = UserSecretsClient().get_secret("GROQ_API_KEY")
except Exception:
    pass

client = Groq(api_key=GROQ_API_KEY)

# ── Cell 3: Signal 1 — Burstiness ────────────────────────────────────────────
def compute_burstiness(text: str) -> float:
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s for s in sentences if len(s.split()) >= 3]
    if len(sentences) < 2:
        return 0.5
    lengths = [len(s.split()) for s in sentences]
    mean = statistics.mean(lengths)
    std  = statistics.stdev(lengths)
    if mean == 0:
        return 0.5
    return min(std / mean, 1.0)

# ── Cell 4: Signal 2 — Linguistic Patterns ───────────────────────────────────
AI_PHRASES = [
    r"\bcertainly\b", r"\babsolutely\b", r"\bof course\b",
    r"\bgreat question\b", r"\bexcellent question\b",
    r"\bI'd be happy to\b", r"\bI would be happy to\b",
    r"\bAs an AI\b", r"\bAs a language model\b",
    r"\bin conclusion\b", r"\bto summarize\b", r"\bin summary\b",
    r"\bto conclude\b", r"\bin closing\b",
    r"\bleverage\b", r"\butilize\b", r"\bfacilitate\b",
    r"\boptimize\b", r"\bsynerg", r"\bseamlessly\b",
    r"\brobust solution\b", r"\bcomprehensive approach\b",
    r"\bfirstly\b.*\bsecondly\b", r"\bfirst and foremost\b",
    r"\bit is worth noting\b", r"\bit is important to note\b",
    r"\bit should be noted\b",
]

def compute_pattern_score(text: str) -> float:
    text_lower = text.lower()
    hits = sum(1 for p in AI_PHRASES if re.search(p, text_lower))
    return min(hits / 3.0, 1.0)

# ── Cell 5: Signal 3 — LLM Judge ─────────────────────────────────────────────
def llm_ai_detection(question: str, answer: str) -> dict:
    prompt = f"""
You are an expert at detecting AI-generated text in interview responses.

Analyze the following interview answer and determine if it was written by:
- A human (natural, conversational, may have imperfections, personal anecdotes)
- An AI (overly structured, formal transitions, no personal voice, generic examples)

Interview Question:
{question}

Candidate Answer:
{answer}

Return ONLY this JSON:
{{
  "is_ai_generated": true or false,
  "confidence": 0.0 to 1.0,
  "signals": ["list of specific signals you noticed"],
  "verdict": "human" or "ai" or "uncertain"
}}

No explanation. No markdown. Just JSON.
"""
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            max_tokens=300,
        )
        raw = response.choices[0].message.content.strip()
        raw = raw.replace("```json", "").replace("```", "").strip()
        return json.loads(raw)
    except Exception as e:
        print(f"[LLM Detection] Error: {e}")
        return {"is_ai_generated": False, "confidence": 0.0, "signals": [], "verdict": "uncertain"}

# ── Cell 6: Combined Detector ─────────────────────────────────────────────────
def detect_ai(question: str, answer: str) -> dict:
    word_count = len(answer.split())
    if word_count < 15:
        return {
            "verdict": "human", "ai_confidence": 0.0,
            "score_multiplier": 1.0, "penalty_reason": None,
            "signals": ["Answer too short to analyze"], "breakdown": {}
        }

    burstiness    = compute_burstiness(answer)
    pattern_score = compute_pattern_score(answer)
    llm_result    = llm_ai_detection(question, answer)

    llm_confidence = llm_result.get("confidence", 0.0)
    if llm_result.get("verdict") == "human":
        llm_confidence = 1.0 - llm_confidence

    ai_confidence = (
        llm_confidence   * 0.50 +
        (1 - burstiness) * 0.30 +
        pattern_score    * 0.20
    )

    if ai_confidence >= 0.70:
        verdict          = "ai"
        score_multiplier = 0.0
        penalty_reason   = (
            f"Response flagged as AI-generated "
            f"(confidence: {ai_confidence:.0%}). "
            f"Signals: {', '.join(llm_result.get('signals', []))}"
        )
    elif ai_confidence >= 0.45:
        verdict          = "uncertain"
        score_multiplier = 0.5
        penalty_reason   = f"Response shows partial AI patterns (confidence: {ai_confidence:.0%}). Score halved."
    else:
        verdict          = "human"
        score_multiplier = 1.0
        penalty_reason   = None

    return {
        "verdict":          verdict,
        "ai_confidence":    round(ai_confidence, 3),
        "score_multiplier": score_multiplier,
        "penalty_reason":   penalty_reason,
        "signals":          llm_result.get("signals", []),
        "breakdown": {
            "burstiness":     round(burstiness, 3),
            "pattern_score":  round(pattern_score, 3),
            "llm_confidence": round(llm_confidence, 3),
        }
    }

# ── Cell 7: Transcript Parser ─────────────────────────────────────────────────
def parse_transcript(transcript: str) -> list[dict]:
    """
    Parses a raw transcript into Q&A pairs.

    Supports formats:
      Q: ...  A: ...
      Interviewer: ...  Candidate: ...
      Question: ... Answer: ...
    """
    patterns = [
        (r"(?:Q|Question|Interviewer)\s*:\s*(.+?)(?=(?:A|Answer|Candidate)\s*:)", 
         r"(?:A|Answer|Candidate)\s*:\s*(.+?)(?=(?:Q|Question|Interviewer)\s*:|$)"),
    ]

    qa_pairs = []

    # Try structured Q/A pattern
    questions = re.findall(
        r"(?:Q|Question|Interviewer)\s*[:\-]\s*(.+?)(?=(?:A|Answer|Candidate)\s*[:\-]|$)",
        transcript, re.DOTALL | re.IGNORECASE
    )
    answers = re.findall(
        r"(?:A|Answer|Candidate)\s*[:\-]\s*(.+?)(?=(?:Q|Question|Interviewer)\s*[:\-]|$)",
        transcript, re.DOTALL | re.IGNORECASE
    )

    if questions and answers:
        for i, (q, a) in enumerate(zip(questions, answers)):
            qa_pairs.append({
                "index":    i + 1,
                "question": q.strip(),
                "answer":   a.strip()
            })
        return qa_pairs

    # Fallback: treat entire transcript as one answer with no question
    return [{
        "index":    1,
        "question": "General interview response",
        "answer":   transcript.strip()
    }]

# ── Cell 8: FastAPI App ───────────────────────────────────────────────────────
app = FastAPI(title="AI Interview Detector")

class TranscriptRequest(BaseModel):
    transcript: str
    candidate_name: Optional[str] = "Unknown"

class QARequest(BaseModel):
    question: str
    answer: str

@app.get("/health")
def health():
    return {"status": "ok", "model": "llama-3.3-70b-versatile"}


@app.post("/detect/transcript")
async def detect_transcript(req: TranscriptRequest):
    """
    Full transcript analysis.
    Parses Q&A pairs and runs detection on each answer.

    Input JSON:
    {
        "transcript": "Q: What is AI? A: AI is ...\nQ: ...",
        "candidate_name": "John Doe"   // optional
    }
    """
    if not req.transcript.strip():
        raise HTTPException(400, "Empty transcript")

    t0 = time.perf_counter()

    # Parse into Q&A pairs
    qa_pairs = parse_transcript(req.transcript)
    if not qa_pairs:
        raise HTTPException(400, "Could not extract Q&A pairs from transcript")

    print(f"→ Analyzing {len(qa_pairs)} Q&A pair(s) for: {req.candidate_name}")

    # Run detection on each answer
    results = []
    ai_count      = 0
    uncertain_count = 0

    for pair in qa_pairs:
        print(f"  [{pair['index']}/{len(qa_pairs)}] Analyzing answer...")
        result = detect_ai(pair["question"], pair["answer"])
        results.append({
            "index":    pair["index"],
            "question": pair["question"],
            "answer":   pair["answer"],
            "detection": result
        })
        if result["verdict"] == "ai":
            ai_count += 1
        elif result["verdict"] == "uncertain":
            uncertain_count += 1
        time.sleep(0.3)  # avoid rate limits

    total_pairs = len(qa_pairs)
    ai_ratio    = ai_count / total_pairs

    # Overall verdict
    if ai_ratio >= 0.5:
        overall_verdict = "ai"
        overall_risk    = "HIGH"
    elif ai_ratio >= 0.25 or uncertain_count / total_pairs >= 0.4:
        overall_verdict = "uncertain"
        overall_risk    = "MEDIUM"
    else:
        overall_verdict = "human"
        overall_risk    = "LOW"

    avg_confidence = sum(r["detection"]["ai_confidence"] for r in results) / total_pairs

    elapsed = round(time.perf_counter() - t0, 3)
    flag = "🚨 ALERT" if overall_verdict == "ai" else ("⚠️ REVIEW" if overall_verdict == "uncertain" else "✅ CLEAN")
    print(f"  Done → {flag}  ai={ai_count}/{total_pairs}  avg_conf={avg_confidence:.2f}  {elapsed}s")

    return JSONResponse({
        "candidate_name":   req.candidate_name,
        "overall_verdict":  overall_verdict,
        "risk_level":       overall_risk,
        "avg_ai_confidence": round(avg_confidence, 3),
        "summary": {
            "total_questions":    total_pairs,
            "ai_answers":         ai_count,
            "uncertain_answers":  uncertain_count,
            "human_answers":      total_pairs - ai_count - uncertain_count,
        },
        "per_question":    results,
        "elapsed_s":       elapsed
    })


@app.post("/detect/single")
async def detect_single(req: QARequest):
    """
    Single Q&A pair detection.

    Input JSON:
    {
        "question": "What is machine learning?",
        "answer":   "Machine learning is..."
    }
    """
    if not req.answer.strip():
        raise HTTPException(400, "Empty answer")

    t0     = time.perf_counter()
    result = detect_ai(req.question, req.answer)
    result["elapsed_s"] = round(time.perf_counter() - t0, 3)
    return JSONResponse(result)


# ── Cell 9: Start localhost.run tunnel ───────────────────────────────────────
def start_tunnel(port: int):
    proc = subprocess.Popen(
        ["ssh", "-o", "StrictHostKeyChecking=no",
         "-R", f"80:localhost:{port}",
         "nokey@localhost.run"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    for line in proc.stdout:
        line = line.decode("utf-8", errors="ignore").strip()
        match = re.search(r"https://[a-zA-Z0-9\-]+\.lhr\.life", line)
        if match:
            url = match.group(0)
            print("\n" + "="*55)
            print(f"  AI Detector API  ->  {url}")
            print(f"  Docs             ->  {url}/docs")
            print(f"  Health           ->  {url}/health")
            print("="*55 + "\n")
            break

threading.Thread(target=start_tunnel, args=(PORT,), daemon=True).start()
time.sleep(5)

# ── Cell 10: Start server (blocks) ───────────────────────────────────────────
nest_asyncio.apply()
config = uvicorn.Config(app, host="0.0.0.0", port=PORT)
server = uvicorn.Server(config)
asyncio.get_event_loop().run_until_complete(server.serve())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 3.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 41.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.7/110.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.9/425.9 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 87.9 MB/s eta 0:00:00:00:01

  AI Detector API  ->  https://73f084fe9efb6c.lhr.life
  Docs             ->  https://73f084fe9efb6c.lhr.life/docs
  Health           ->  https://73f084fe9efb6c.lhr.life/health



INFO:     Started server process [55]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


INFO:     38.183.101.34:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     38.183.101.34:0 - "GET /openapi.json HTTP/1.1" 200 OK
